# 06 - Recommandations métiers et synthèse finale

**Projet ST2MLE - Analyse des offres d'emploi en France**

Ce notebook clôt le TP en transformant les résultats des notebooks 03, 04 et 05 en **recommandations métier concrètes**. L'objectif n'est plus de comparer des algorithmes, mais de répondre à une question simple : **comment exploiter les modèles dans un contexte opérationnel ?**

La logique suivie est la suivante :
1. rappeler les meilleurs constats du TP ;
2. traduire ces résultats en décisions applicables ;
3. proposer une stratégie de déploiement réaliste ;
4. identifier les limites et les améliorations possibles.

## 1. Synthèse rapide des résultats obtenus

### A. Modélisation numérique du salaire
Les modèles à base d'arbres ont montré que le salaire peut être estimé à partir de variables structurelles comme l'expérience, le type de contrat, le métier et la localisation.

En pratique, le meilleur modèle est celui qui combine le **$R^2$ le plus élevé** et la **MAE la plus faible**. Le **Random Forest** domine ces deux métriques, alors c'est lui qu'il faut retenir comme modèle principal pour le salaire.

### B. Modélisation textuelle
Pour les tâches de classification du métier et du contrat, les approches classiques restent très solides : **TF-IDF + Régression Logistique** offre en général le meilleur compromis entre performance, coût de calcul et interprétabilité.

### C. Vectorisation sémantique
Les méthodes plus avancées ont confirmé une hiérarchie claire : **CamemBERT** surpasse Doc2Vec sur la qualité sémantique, tandis que Doc2Vec reste plus fragile sur un corpus de taille limitée.

### D. Enseignement central
Le meilleur modèle n'est pas le même selon l'usage :
- pour le **salaire**, on retient le modèle tabulaire qui maximise $R^2$ tout en minimisant la MAE ;
- pour la **classification textuelle opérationnelle**, on privilégie TF-IDF + Régression Logistique ;
- pour les usages sémantiques à plus forte valeur ajoutée, on réserve CamemBERT aux cas où le coût et la latence sont acceptables.

In [ ]:
import pandas as pd

synthese = pd.DataFrame([
    {
        'Besoin': 'Estimer un salaire',
        'Meilleure famille de modèles': 'Random Forest si ses scores dominent les autres',
        'Usage recommandé': 'Estimation robuste d\'une fourchette de salaire et détection des offres atypiques',
    },
    {
        'Besoin': 'Classer métier / contrat',
        'Meilleure famille de modèles': 'TF-IDF + Régression Logistique',
        'Usage recommandé': 'Auto-tagging fiable et interprétable des offres à la publication',
    },
    {
        'Besoin': 'Comparer le sens des textes',
        'Meilleure famille de modèles': 'CamemBERT',
        'Usage recommandé': 'Recherche sémantique, matching CV-offre, enrichissement de qualité',
    },
    {
        'Besoin': 'Baseline rapide',
        'Meilleure famille de modèles': 'Naive Bayes / BoW',
        'Usage recommandé': 'Prototype ou test initial, mais pas le meilleur choix final',
    },
])
display(synthese)

## 2. Recommandations métier par cas d'usage

### A. Automatiser le classement des offres
Le premier gain immédiat est l'automatisation du **tagging** des annonces. Dès qu'une offre est saisie, le modèle peut prédire le métier et le type de contrat sans intervention humaine.

Recommandation : déployer en production un couple **TF-IDF + Régression Logistique** pour la classification du métier et du contrat. Ce choix est pertinent car il est :
- rapide à exécuter ;
- facile à maintenir ;
- explicable par les poids des mots ;
- suffisamment performant pour un usage métier quotidien.

### B. Aider à la fixation d'un salaire
Le modèle de régression du salaire peut servir à proposer une **fourchette indicative** lors de la rédaction d'une annonce. C'est utile pour éviter les offres incohérentes, améliorer l'attractivité et comparer les postes d'un même secteur.

Recommandation : utiliser le **meilleur modèle mesuré sur votre tableau de résultats**. La Random Forest ayant à la fois une MAE plus faible et un $R^2$ plus élevé que les autres, elle doit être le moteur principal, avec un contrôle humain sur les cas extrêmes. Le modèle doit être présenté comme un outil d'aide à la décision, pas comme un arbitre unique.

### C. Améliorer la recherche et le matching
Les vecteurs issus de **CamemBERT** sont particulièrement adaptés à la recherche sémantique : ils capturent mieux le sens global d'une offre qu'une simple liste de mots-clés.

Recommandation : utiliser CamemBERT pour :
- rapprocher automatiquement les CV et les offres ;
- suggérer des offres similaires ;
- enrichir un moteur de recommandation interne.

### D. Garder une solution frugale pour le temps réel
Si la plateforme doit traiter beaucoup de requêtes en continu, il faut conserver une solution légère. Dans ce cas, **TF-IDF + Régression Logistique** reste le meilleur compromis entre coût, vitesse et qualité.

Conclusion opérationnelle :
- **temps réel** -> modèles classiques ;
- **batch / haute qualité** -> CamemBERT ;
- **score salaire** -> le modèle tabulaire le plus performant sur vos métriques, souvent Random Forest si elle domine $R^2$ et MAE.

## 3. Recommandations par acteur du projet

### Pour une plateforme d'emploi
- standardiser les champs d'annonce à partir des prédictions du modèle ;
- déclencher une alerte si le salaire estimé est très éloigné de la médiane du marché ;
- proposer des offres similaires grâce à l'embedding sémantique.

### Pour les recruteurs
- utiliser le modèle de salaire pour calibrer l'offre avant publication ;
- vérifier les prédictions de métier/contrat pour éviter les erreurs de catégorisation ;
- améliorer la visibilité des annonces en enrichissant les mots-clés et la structure du texte.

### Pour les candidats
- mieux comparer les offres à salaire équivalent ;
- identifier plus vite les contrats et métiers pertinents ;
- accéder à des recommandations sémantiques plus proches de leur profil.

### Pour un responsable data / produit
- mettre en place un pipeline à deux niveaux : modèle léger pour la production courante, modèle sémantique plus lourd pour les traitements différés ;
- suivre des KPI simples : F1 macro pour les classes, MAE/RMSE pour le salaire, taux d'auto-tagging réussi, temps de réponse.

## 4. Limites et points de vigilance

Les résultats du TP sont exploitables, mais ils ne doivent pas être sur-interprétés. Les principales limites sont les suivantes :

- les données de salaire sont parfois approximatives ou incomplètes ;
- la qualité des catégories dépend du nettoyage et du regroupement réalisés en amont ;
- les classes restent déséquilibrées pour certaines cibles ;
- les textes d'offres peuvent contenir du bruit, des formulations commerciales ou des biais de rédaction ;
- CamemBERT apporte de meilleures performances, mais avec un coût de calcul plus élevé.

Recommandation de gouvernance : surveiller la dérive des données dans le temps et réentraîner les modèles régulièrement, surtout si le marché de l'emploi évolue rapidement.

## 5. Conclusion finale

Le TP montre qu'un projet de machine learning sur des données d'offres d'emploi françaises peut produire de la valeur à trois niveaux :

1. **Automatiser la structuration des offres** avec une classification textuelle robuste ;
2. **Aider à estimer le salaire** avec un modèle tabulaire performant ;
3. **Améliorer l'expérience de recherche et de matching** grâce à des vecteurs sémantiques plus riches.

En synthèse :
- **TF-IDF + Régression Logistique** est le meilleur choix pour une solution simple, rapide et explicable ;
- **Boosting** est le meilleur candidat pour la prédiction du salaire ;
- **CamemBERT** doit être réservé aux usages où la qualité sémantique justifie un coût supérieur.

Cette architecture permet de proposer une solution réaliste, utile et directement valorisable en contexte métier.